环境：Apache Polaris (REST Catalog)-1.3.0 + Spark SQL-3.5.8 + Kyuubi-1.10.3 + MinIO

## 概念介绍
### 为什么需要分支与标签
在没有分支（Branch）和标签（Tag）功能之前，管理数据湖的版本就像走钢丝，充满了风险和低效：

+ **痛点一：开发测试会弄脏生产数据**
    - 场景：你想测试一个新的 ETL 逻辑，或者修复一个数据 bug。
    - 后果：你只能直接在生产表上跑 `INSERT` 或 `UPDATE`。如果逻辑错了，生产表就被污染了，你必须手忙脚乱地回滚，甚至可能因为快照太多而找不到干净的还原点。
+ **痛点二：无法稳定地标记重要时刻**
    - 场景：每天凌晨跑完批处理，你想标记这是周一的正式数据。
    - 后果：你只能记下当时的 `snapshot_id`（一串长长的数字）。过几天想找回这个状态，得去翻日志找那串数字，极易记错或丢失。
+ **痛点三：多任务并发冲突**
    - 场景：两个团队想基于同一份数据做不同的实验。
    - 后果：他们必须复制整份数据（浪费存储）或者排队等待（降低效率）。没有隔离的开发环境，协作极其困难。

**Iceberg 引入了类似 Git 的 分支 (Branch) 和 标签 (Tag) 机制：**

+ 分支 (Branch)：是一个可移动的指针。你可以在上面随意写入、测试，完全不影响主分支（main）。测试成功后，可以合并回主线；失败了直接删除分支，毫发无损。
+ 标签 (Tag)：是一个固定不变的指针。一旦打上标签（如 `v2026-03-14`），它永远指向那个特定的快照。无论后续数据怎么变，通过这个标签都能瞬间回到那个时刻。

---

## 实战案例：电商订单数据开发与发布
场景：我们有一张订单表 `dataspire_catalog.db_dev.orders`。

1. 发布阶段：将清洗好的 T+1 数据打上 `v20260314_release` 标签，作为生产基准。
2. 开发阶段：发现部分订单状态逻辑有误，创建 `fix_order_status` 分支进行修复验证。
3. 合并阶段：验证无误后，将分支数据合并回主表，并清理分支。

### 初始化数据
首先，创建表并模拟初始数据加载。

```sql
-- 1. 创建订单表 (使用 Iceberg 格式，按天分区)
CREATE TABLE IF NOT EXISTS dataspire_catalog.db_dev.orders (
    order_id BIGINT COMMENT '订单ID',
    user_id BIGINT COMMENT '用户ID',
    amount DOUBLE COMMENT '金额',
    status STRING COMMENT '订单状态',
    created_at TIMESTAMP COMMENT '创建时间',
    dt STRING COMMENT '分区字段'
) USING iceberg
PARTITIONED BY (dt);

-- 2. 插入初始数据 (模拟 2026-03-13 的数据)
INSERT
INTO
    dataspire_catalog.db_dev.orders
VALUES
    (1001, 501, 99.50, 'PAID', TIMESTAMP('2026-03-13 10:00:00'), '2026-03-13')
  , ( 1002, 502, 200.00, 'PENDING', TIMESTAMP('2026-03-13 11:30:00'), '2026-03-13')
  , (1003, 503, 50.25, 'PAID', TIMESTAMP('2026-03-13 12:15:00'), '2026-03-13')
  , ( 1004, 504, 300.00, 'PENDING', TIMESTAMP('2026-03-13 14:20:00'), '2026-03-13')
;

-- 3. 查看当前快照历史，确认数据已提交
SELECT
    made_current_at     AS made_current_at
  , snapshot_id         AS snapshot_id
  , parent_id           AS parent_id
  , is_current_ancestor AS parent_id
FROM
    dataspire_catalog.db_dev.orders.history
WHERE
    1 = 1
ORDER BY
    made_current_at DESC
LIMIT 20
;
```

---

### 标签操作：锁定生产版本
数据核对无误后，我们需要创建一个标签，确保即使未来数据被误删或覆盖，也能随时回溯到这个版本。

```sql
-- 1. 基于当前最新快照创建标签 'v20260314_release'
-- RETAIN 365 DAYS 表示该标签及其指向的快照至少保留 365 天，防止被过期清理
ALTER TABLE dataspire_catalog.db_dev.orders CREATE TAG v20260314_release RETAIN 365 DAYS;
```

如何使用标签进行时间旅行查询：

```sql
-- 场景：假设主表数据后来被错误修改，我们需要审计原始数据
-- 语法：VERSION AS OF 'tag_name'
SELECT
    *
FROM
    dataspire_catalog.db_dev.orders VERSION AS OF 'v20260314_release'
WHERE
    1 = 1
  AND
    `STATUS` = 'PENDING'
;

-- 也可以基于快照 ID 查询 (效果等价)
-- SELECT * FROM dataspire_catalog.db_dev.orders VERSION AS OF <snapshot_id>;
```

---

### 分支操作
业务方反馈 `PENDING` 超过 2 小时的订单应自动标记为 `CANCELLED`。为了不影响生产查询，我们在分支上进行验证。

```sql
-- 1. 基于当前主表创建开发分支 'fix_order_status'
-- 默认基于最新快照创建，也可指定 AS OF VERSION <id>
ALTER TABLE dataspire_catalog.db_dev.orders CREATE BRANCH `fix_order_status` RETAIN 7 DAYS; -- 开发分支通常保留时间较短

-- 2. 向分支写入修正后的数据
-- 关键点：表名后缀加上 .branch_name
-- 逻辑：读取分支当前数据 -> 转换状态 -> 覆盖写回分支
INSERT OVERWRITE dataspire_catalog.db_dev.orders.fix_order_status
SELECT
    order_id   AS order_id
  , user_id    AS user_id
  , amount     AS amount
  , CASE
        WHEN status = 'PENDING' AND
             created_at < CURRENT_TIMESTAMP() - INTERVAL 2 HOURS
            THEN 'CANCELLED'
        ELSE status
        END    AS status
  , created_at AS created_at
  , dt         AS dt
FROM
    dataspire_catalog.db_dev.orders VERSION AS OF 'fix_order_status' -- 从分支读取
WHERE
    1 = 1 AND dt = '2026-03-13'
;

-- 3. 验证分支数据 (隔离环境)
-- 此时应看到符合条件的 PENDING 变成了 CANCELLED
SELECT order_id, status, created_at 
FROM dataspire_catalog.db_dev.orders.fix_order_status 
WHERE order_id IN (1002, 1004);

-- 4. 对比主表数据 (生产环境未受影响)
-- 此时主表仍应为 PENDING
SELECT order_id, status, created_at 
FROM dataspire_catalog.db_dev.orders 
WHERE order_id IN (1002, 1004);
```

---

### 分支合并与清理
Iceberg SQL 标准目前不直接支持`ALTER TABLE ... MERGE BRANCH` 语法。最佳实践是通过 `INSERT OVERWRITE` 将分支验证后的数据应用到主表（针对特定分区或全量）。

#### 场景 A：将分支变更应用到主表
```sql
-- 将分支中验证过的数据覆盖回主表的对应分区
-- 这是一种“快进”(Fast-forward) 或 “拣选”(Cherry-pick) 的手动实现
INSERT OVERWRITE dataspire_catalog.db_dev.orders
PARTITION (dt = '2026-03-13')
SELECT order_id, user_id, amount, status, created_at
FROM dataspire_catalog.db_dev.orders.fix_order_status
WHERE dt = '2026-03-13';

-- 再次检查主表，确认状态已更新
SELECT order_id, status 
FROM dataspire_catalog.db_dev.orders 
WHERE order_id IN (1002, 1004);
```

#### 场景 B：清理分支
合并完成后，分支不再需要，应删除以简化元数据。

```sql
-- 删除分支
ALTER TABLE dataspire_catalog.db_dev.orders 
DROP BRANCH fix_order_status;

-- 确认分支已删除
SELECT branch_name FROM dataspire_catalog.db_dev.orders.branches;
```

---

### 标签管理
```sql
-- 如果标签不再需要（例如过了合规保留期），可以删除
-- 注意：删除标签不会立即删除底层数据文件，除非快照过期且被 GC 清理
ALTER TABLE dataspire_catalog.db_dev.orders 
DROP TAG v20260314_release;
```

---

## 最佳实践
### 命名规范
遵循清晰的命名约定有助于团队协作和自动化脚本编写。

+ Branch: `<type>/<description>-<date>`
    - 示例: `feat/add-user-column-20260314`, `fix/order-status-bug`, `etl/retry-load-001`
    - 避免: `test`, `new`, `branch1`
+ Tag: `<scope>-<version/date>`
    - 示例: `prod-v2.1.0`, `audit-2026-q1`, `golden-20260314`

### 设置合理的保留策略 
在生产环境中，快照过期清理（Expire Snapshots）是定期运行的。必须为 Branch 和 Tag 设置 `RETAIN` 时间，否则它们可能因指向的快照被过早清理而失效，或者导致无用快照无限堆积。

+ 开发分支: `RETAIN 3 DAYS` 或 `RETAIN 7 DAYS`。长生命周期的分支应定期 rebase 或重建。
+ 生产标签: `RETAIN 365 DAYS` 或更久，取决于审计合规要求。
+ SQL 示例:

```sql
ALTER TABLE dataspire_catalog.db_dev.orders CREATE BRANCH temp_exp RETAIN 3 DAYS;
ALTER TABLE dataspire_catalog.db_dev.orders CREATE TAG yearly_audit_2026 RETAIN 2555 DAYS; -- 7年
```

---

### 性能与成本优化
+ 元数据轻量化: Branch/Tag 本身只存储指针，几乎无成本。成本主要来自于它们阻止了旧快照的清理。如果一个 Tag 保留了 1000 个快照，那么这 1000 个快照对应的数据文件都不能被 GC 删除。
+ 定期清理: 建立自动化作业，定期检查并删除不再需要的 Branch。
+ 查询性能: 查询 Tag/Branch 的历史数据性能与查询当前表一致（O(1) 元数据查找）。但如果回溯极久远的快照，且元数据链过长，可能会有轻微开销（Iceberg 元数据树结构通常能很好处理此问题）。

### 并发与隔离性
+ 写隔离: 向 `table.branch_A` 写入完全不会影响 `table.main` 或 `table.branch_B` 的读取。这是实现 CI/CD 流水线的关键。
+ 冲突检测: Iceberg 使用乐观锁。如果两个任务同时向同一个分支提交，后提交的任务会失败（检测到快照冲突），需要重试。但这不会波及其他分支。

### 常见误区警示
1. 误区: Branch 是数据的物理副本。
+ 真相: Branch 只是元数据指针。只有在分支上执行写操作产生新文件时，才会占用额外存储。未修改的文件在主表和分支间是共享的。
2. 误区: 可以直接 `INSERT INTO table.tag_name`。
+ 真相: Tag 是只读的。尝试写入会抛出异常。如需基于 Tag 开发，应先基于该 Tag 创建一个新的 Branch。
3. 误区: 删除 Branch 就会删除数据。
+ 真相: 删除 Branch 只是删除了指针。如果该 Branch 产生的快照还被其他 Tag 或 Branch 引用，或者未达到过期时间，数据文件不会被删除。需运行 `EXPIRE_SNAPSHOTS` 过程才能真正清理文件。

---

## 总结速查表
| 创建分支 | `ALTER TABLE <table> CREATE BRANCH <name> [AS OF VERSION <id>] RETAIN <N> DAYS;` |
| --- | --- |
| 创建标签 | `ALTER TABLE <table> CREATE TAG <name> [AS OF VERSION <id>] RETAIN <N> DAYS;` |
| 查询分支 | `SELECT * FROM <table>.<branch_name>;` |
| 查询标签 | `SELECT * FROM <table> VERSION AS OF '<tag_name>';` |
| 写入分支 | `INSERT INTO <table>.<branch_name> SELECT ...;`<br/>`INSERT OVERWRITE <table>.<branch_name> SELECT ...;` |
| 删除分支 | `ALTER TABLE <table> DROP BRANCH <name>;` |
| 删除标签 | `ALTER TABLE <table> DROP TAG <name>;` |
| 列出所有分支 | `SELECT * FROM <table>.branches;` |
| 列出所有标签 | `SELECT * FROM <table>.tags;` |
| 查看历史快照 | `SELECT * FROM <table>.history;` |
